In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# matplotlib and seaborn for plotting
import matplotlib.pyplot as plt
import seaborn as sns

# text processing libraries
import re #regular expressions, die wir später beim Processing brauchen
import string
import nltk

# Suppress warnings 
import warnings
warnings.filterwarnings('ignore')

In [3]:
pip install --upgrade pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install openpyxl --upgrade

In [28]:
all_messages = df["Content"]
all_messages.head()

ExperimentID
2020_12     \nDear Mr. Williams, \nI am happy to be respo...
2020_12    Dear Mr. Brown, \nI am pleased to hear from yo...
2020_12    Dear Mr. Williams, \nthank you for your quick ...
2020_12    Dear Mr. Brown, \nThe concessions of your last...
2020_12    Dear Mr. Williams,  \nthank you very much for ...
Name: Content, dtype: object

In [29]:
#Cleaning the Corpus 
'''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''

def clean_text(message):
    #lowercasing
    message = message.lower()
    
    #regular expressions, gerne mal nachschauen wie das funktioniert
    message = re.sub('\[.*?\]', '', message)
    message = re.sub('https?://\S+|www\.\S+', '', message)
    message = re.sub('<.*?>+', '', message)
    message = re.sub('[%s]' % re.escape(string.punctuation), '', message)
    message = re.sub('\n', '', message)
    message = re.sub('\w*\d\w*', '', message)
    
    return message

# Applying the cleaning function to the content column
df['Content'] = df['Content'].apply(lambda x: clean_text(x))

# Let's take a look at the updated messages
df['Content'].head()

ExperimentID
2020_12     dear mr williams i am happy to be responsible...
2020_12    dear mr brown i am pleased to hear from youtha...
2020_12    dear mr williams thank you for your quick and ...
2020_12    dear mr brown the concessions of your last off...
2020_12    dear mr williams  thank you very much for your...
Name: Content, dtype: object

## 2.2) Tokenization

In [31]:
# Bereinigte Sätze in Word-Tokens aufgerteilt

tokenizer = nltk.tokenize.RegexpTokenizer(r'\w+')
df['Content'] = df['Content'].apply(lambda x: tokenizer.tokenize(x))
df['Content'].head()

ExperimentID
2020_12    [dear, mr, williams, i, am, happy, to, be, res...
2020_12    [dear, mr, brown, i, am, pleased, to, hear, fr...
2020_12    [dear, mr, williams, thank, you, for, your, qu...
2020_12    [dear, mr, brown, the, concessions, of, your, ...
2020_12    [dear, mr, williams, thank, you, very, much, f...
Name: Content, dtype: object

## 2.3) Stopword Removal

In [33]:
from nltk.corpus import stopwords

#stopword removal
#Often we want to remove stopwords when we want to keep the "gist" of the document/sentence.
stopwords_en = stopwords.words('english')
print(stopwords_en)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [34]:
 #Vordefinierte Stopword-Liste von NLTK vorhanden
# Removing stopwords belonging to english language
    
def remove_stopwords(text):
    words = [w for w in text if w not in stopwords.words('english')]
    return words

df['Content'] = df['Content'].apply(lambda x : remove_stopwords(x))
df['Content'].head()

ExperimentID
2020_12    [dear, mr, williams, happy, responsible, negot...
2020_12    [dear, mr, brown, pleased, hear, youthank, las...
2020_12    [dear, mr, williams, thank, quick, cooperative...
2020_12    [dear, mr, brown, concessions, last, offer, sh...
2020_12    [dear, mr, williams, thank, much, response, ha...
Name: Content, dtype: object

## 2.4) Token Normalisation

Short Example for Stemming & Lemmatization - see Theory for Details

In [37]:
#Normalization of tokens ---> explain Stemming and Lemmatization in Theory
#Example to see one example in Stemming and Lemmatization: 

text = "I welcome you all in the AI in Action course of the intelligent information systems chair"

tokenizer = nltk.tokenize.TreebankWordTokenizer()
tokens = tokenizer.tokenize(text)

# Stemmer
stemmer = nltk.stem.PorterStemmer()
print("Stemming the sentence: ", " ".join(stemmer.stem(token) for token in tokens))

# Lemmatizer
lemmatizer=nltk.stem.WordNetLemmatizer()
print("Lemmatizing the sentence: ", " ".join(lemmatizer.lemmatize(token) for token in tokens))

Stemming the sentence:  i welcom you all in the ai in action cours of the intellig inform system chair
Lemmatizing the sentence:  I welcome you all in the AI in Action course of the intelligent information system chair


In [38]:
#Lemmatization Platzhalter --> nicht angewandt
'''
It is important to note here that stemming and lemmatization sometimes donot necessarily improve results as at times we donot want to trim words but rather preserve their original form. Hence their usage actually differs from problem to problem. For this problem, I will not use these techniques.
'''

# After preprocessing, the text format
# wörter wieder zusammensetzen
def combine_text(list_of_text):
    '''Takes a list of text and combines them into one large chunk of text.'''
    combined_text = ' '.join(list_of_text)
    return combined_text

df['Content'] = df['Content'].apply(lambda x : combine_text(x))

#DIESES DF[CONTENT] WERDEN WIR IMMER WIEDER EINLESEN; DAS NEHMEN WIR ALS DATENBASIS AN
df['Content']


ExperimentID
2020_12    dear mr williams happy responsible negotiation...
2020_12    dear mr brown pleased hear youthank last messa...
2020_12    dear mr williams thank quick cooperative respo...
2020_12    dear mr brown concessions last offer showed ri...
2020_12    dear mr williams thank much response happy fou...
                                 ...                        
2020_06    dear chis meyer thank latest offer points comp...
2020_06    dear alex kramerthank much latest offer happy ...
2020_06    dear chris meyerthank fast answer happy found ...
2020_06    dear alex krameri still interested coming join...
2020_06    dear chris meyer im sorry accept less mei unde...
Name: Content, Length: 1150, dtype: object

In [39]:
df.head()

,NegotiationID,SenderID,ReceiverID,ActionType,NegoOutcome,Content,Length,Word_count of negotiation message,NegoOutcomeLabel
ExperimentID,,,,,,,,,
2020_12,65172,65159,65162,Offer,FinalAccept,dear mr williams happy responsible negotiation...,1250,227,1
2020_12,65172,65162,65159,Counteroffer,FinalAccept,dear mr brown pleased hear youthank last messa...,1254,219,1
2020_12,65172,65159,65162,Counteroffer,FinalAccept,dear mr williams thank quick cooperative respo...,1467,252,1
2020_12,65172,65162,65159,Counteroffer,FinalAccept,dear mr brown concessions last offer showed ri...,857,157,1
2020_12,65172,65159,65162,Counteroffer,FinalAccept,dear mr williams thank much response happy fou...,686,124,1
